# Разбиение данных

Вход: `data/processed/`. Выход: `data/splits/seed_42/<dataset>/`.

- **Amazon:** случайное разбиение buy-пар 80/10/10, `SEED = 42`. Обратные пары вместе;
  validation/test исключаются из обоих отношений train.
- **Retailrocket и REES46:** глобальное временное разбиение 80/10/10.
  Сессии и транзакции на границах исключаются целиком; повторные связи в разные периоды допустимы.
- Train содержит buy/view; validation/test — только buy известных товаров.
  `known_user` отмечает известных пользователей для оценки 4D.

Доли после фильтрации могут отличаться от заданных. Seed влияет только на случайный split Amazon.

In [1]:
%pip install -q pandas numpy


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = Path.cwd()
DATA = ROOT / "data" / "processed"
SEED = 42
TRAIN_SHARE, VALID_SHARE = 0.8, 0.1
OUT = ROOT / "data" / "splits" / f"seed_{SEED}"
CHUNK_SIZE = 250_000
SPLITS = ["train", "validation", "test"]

## Общие функции

In [3]:
def pair_keys(frame):
    """Возвращает одинаковый ключ для пары A–B и обратной B–A."""
    left, right = frame.source_id, frame.target_id
    return pd.MultiIndex.from_arrays([left.where(left <= right, right), right.where(left <= right, left)])


def split_amazon():
    """Делит Amazon по buy-парам и исключает holdout из обучающих buy/view."""
    folder = OUT / "amazon"
    folder.mkdir(parents=True, exist_ok=True)
    pairs = pd.read_csv(DATA / "amazon" / "pairs.csv.gz", dtype="string")

    buy = pairs[pairs.relation.eq("buy")]
    keys = pair_keys(buy).unique().sort_values()
    order = np.random.default_rng(SEED).permutation(len(keys))

    first, second = int(len(keys) * TRAIN_SHARE), int(len(keys) * (TRAIN_SHARE + VALID_SHARE))
    valid_keys, test_keys = keys[order[first:second]], keys[order[second:]]
    holdout = valid_keys.append(test_keys)
    train = pairs[~pair_keys(pairs).isin(holdout)]
    known_items = pd.Index(pd.concat([train.source_id, train.target_id]).unique())

    frames = {"train": train, "validation": buy[pair_keys(buy).isin(valid_keys)],
              "test": buy[pair_keys(buy).isin(test_keys)]}
    for name, frame in frames.items():
        if name != "train":
            frame = frame[frame.source_id.isin(known_items) & frame.target_id.isin(known_items)]
        frame.to_csv(folder / f"{name}.csv.gz", index=False)
        print(name, len(frame))

    holdout.to_frame(index=False, name=["source_id", "target_id"]).to_csv(folder / "holdout_pairs.csv.gz", index=False)
    pd.DataFrame({"item_id": known_items.sort_values()}).to_csv(folder / "train_items.csv.gz", index=False)


In [4]:
def temporal_groups(dataset):
    """Назначает временные части событиям, исключая целиком пограничные и связанные группы."""
    columns = ["timestamp", "user_id", "item_id", "session_id"]
    if dataset == "retailrocket":
        columns.append("transactionid")
    
    events = pd.read_csv(DATA / dataset / "events.csv.gz", usecols=columns, dtype="string")
    events["timestamp"] = pd.to_datetime(events.timestamp, utc=True, errors="coerce")
    invalid = events[["timestamp", "user_id", "item_id", "session_id"]].isna().any(axis=1)
    times = events.loc[~invalid, "timestamp"].sort_values().reset_index(drop=True)

    cut1 = times.iloc[int(len(times) * TRAIN_SHARE)]
    cut2 = times.iloc[int(len(times) * (TRAIN_SHARE + VALID_SHARE))]
    if cut1 >= cut2:
        raise ValueError("Временные границы совпали; выберите другой период или доли")
    
    events["split"] = (events.timestamp.ge(cut1).astype(int) + events.timestamp.ge(cut2).astype(int))
    events.loc[invalid, "split"] = -1
    group_columns = ["session_id"] + (["transactionid"] if dataset == "retailrocket" else [])
    while True:
        previous = int(events.split.eq(-1).sum())
        for column in group_columns:
            grouped = events.groupby(["user_id", column])["split"]
            low, high = grouped.transform("min"), grouped.transform("max")
            bad = low.notna() & (low.lt(0) | low.ne(high))
            events.loc[bad, "split"] = -1

        if int(events.split.eq(-1).sum()) == previous:
            break
    kept = events[events.split.ge(0)].copy()
    if set(kept.split.unique()) != {0, 1, 2}:
        raise ValueError("После исключения групп одна из временных частей пуста")
    
    mappings = {}
    for relation, column in [("view", "session_id"),
                              ("buy", "transactionid" if dataset == "retailrocket" else "session_id")]:
        groups = kept.dropna(subset=[column])[["user_id", column, "split"]].drop_duplicates()
        mappings[relation] = groups.rename(columns={column: "group_id"}).assign(relation=relation)
    train = kept[kept.split.eq(0)]
    metadata = {"validation_start": str(cut1), "test_start": str(cut2),
                "events_removed": int(events.split.eq(-1).sum())}
    return pd.concat(mappings.values(), ignore_index=True), set(train.item_id), set(train.user_id), metadata


def split_events(dataset):
    """Сохраняет пары по временным частям; для оценки оставляет покупки известных товаров."""
    folder = OUT / dataset
    folder.mkdir(parents=True, exist_ok=True)

    groups, known_items, known_users, metadata = temporal_groups(dataset)
    
    columns = pd.read_csv(DATA / dataset / "pairs.csv.gz", nrows=0).columns.tolist() + ["known_user"]
    counts = dict.fromkeys(SPLITS, 0)
    for name in SPLITS:
        pd.DataFrame(columns=columns).to_csv(folder / f"{name}.csv.gz", index=False)

    for chunk in pd.read_csv(DATA / dataset / "pairs.csv.gz", dtype="string", chunksize=CHUNK_SIZE):
        chunk = chunk.merge(groups, on=["user_id", "group_id", "relation"], how="inner", validate="many_to_one")
        for i, name in enumerate(SPLITS):
            part = chunk[chunk.split.eq(i)].drop(columns="split").copy()
            if i > 0:
                part = part[part.relation.eq("buy") & part.source_id.isin(known_items)
                            & part.target_id.isin(known_items)].copy()
            part["known_user"] = part.user_id.isin(known_users)
            part.to_csv(folder / f"{name}.csv.gz", mode="a", header=False, index=False)
            counts[name] += len(part)

    pd.DataFrame({"item_id": sorted(known_items)}).to_csv(folder / "train_items.csv.gz", index=False)
    
    metadata.update(seed=SEED, train_share=TRAIN_SHARE, validation_share=VALID_SHARE)
    (folder / "split_info.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))

    print(dataset, counts)
    print("Границы:", metadata["validation_start"], metadata["test_start"])


## Amazon Musical Instruments

In [5]:
split_amazon()

train 312439
validation 8195
test 8243


## Retailrocket

In [6]:
split_events("retailrocket")

retailrocket {'train': 1000133, 'validation': 944, 'test': 747}
Границы: 2015-08-18 04:21:50.278000+00:00 2015-09-02 17:47:20.695000+00:00


## REES46 Cosmetics Events

In [7]:
split_events("rees46_cosmetics")

rees46_cosmetics {'train': 26993228, 'validation': 1121086, 'test': 1015086}
Границы: 2020-01-31 23:40:01+00:00 2020-02-14 20:06:22+00:00


## REES46 Electronics Events

In [8]:
split_events("rees46_electronics")

rees46_electronics {'train': 322033, 'validation': 926, 'test': 740}
Границы: 2021-01-30 10:17:09+00:00 2021-02-13 11:50:32+00:00


## Результат

Для каждого набора сохраняются `train.csv.gz`, `validation.csv.gz`, `test.csv.gz` и `train_items.csv.gz`.

Дополнительно:
- Amazon: `holdout_pairs.csv.gz` для исключения holdout из обучающих negatives.
- Retailrocket и REES46: `split_info.json` с границами времени, параметрами и числом исключённых событий.
